# Chronos 2 For Time Series With Fixed Data
Improves on an earlier notebook by building around fixed data: issues with event to place attribution have been resolved. Additional country-specific columns have also been added, as has election information. 

Builds a model with Amazon's Chronos-2 pretrained model.
Heavily adapted from https://huggingface.co/amazon/chronos-2 and https://colab.research.google.com/github/amazon-science/chronos-forecasting/blob/main/notebooks/chronos-2-quickstart.ipynb#scrollTo=39de5d7e with some reference to https://deepwiki.com/amazon-science/chronos-forecasting/1-overview

Chronos 2 was selected as it supports cross training. Serious difficulties overcome with Chronos 1 and Chronos Bolt. Previous attempt with TimeSeriesTransformerForPrediction also failed as couldn; get the model to make predictions despite being able to train it. That model was selected for ONNX export potential (ease of use from within C#).

As per Amazon's HuggingFace model page: "Chronos-2 is a 120M-parameter, encoder-only time series foundation model for zero-shot forecasting." The encoder learns representations of the past. As a foundation model, it is trained on a large number of time series datasets across different domains, therefore should generalise out to this problem. As a model with the ability to do zero-shot forecasting, it can make predictions without havign to be trained on our dataset, i.e, we are providing it context of the past data to allow a prediction, not adjusting internal model weights as per our dataset- making the predictions means it does not actually learn anything new, it is acting as block box data in -> maths -> predictions out.

Chronos 2 improves over prior Chronos models by adding Cross-learning across items, Multivariate Forecasting, Past-only (real/categorical) covariates, and better Known future (real/categorical) covariates support. It also has a larger context window of 8192 comapred to Chronos-Bolt (2048) and Chronos 1 (512) and support for much longer prediction steps (1024)

Without a decoder, Chronos 2 is not capable of generating new tokens, or their probabilities. Instead we get the quartiles (often 0.1, 0.5, 0.9).

## Load Dataset

In [1]:
import pandas as pd
df_context_full = pd.read_csv("Dataset Preprocessed\eastafrica3_2026-09-20_Train.csv")
df_future_full = pd.read_csv("Dataset Preprocessed\eastafrica3_2026-09-20_Test.csv")

### Minor Preprocessing

In [2]:
#format the perdiodStart as proper date:
df_context_full['periodStart'] = pd.to_datetime(df_context_full['periodStart'],  format="%Y-%m-%d")
df_future_full['periodStart'] = pd.to_datetime(df_future_full['periodStart'],  format="%Y-%m-%d")

In [3]:

#remove targets and unknowable values from future covariates (we'll remove the target from the subsamples when we create them). 
df_future_full_columns = ["id", "name",	"country",	"latitude",	"longitude",	"minBorderDistanceKm",	"minCapitalDistanceKm",	"periodStart", "conflict_indicator", 'is_voting']
df_future_subset = df_future_full[df_future_full_columns]

## Validation

In [6]:
#Context:
context_counts = (
    df_context_full.groupby("id")
      .size()
      .reset_index(name="count")
)

bad_ids_context = context_counts.query("count != 127")

print("Bad IDs in df_future_full:", bad_ids_context["id"].tolist())

# Show counts for each bad ID
print("Counts for bad IDs (df_future_full):")
print(bad_ids_context)

#Future:
future_counts = (
    df_future_full.groupby("id")
      .size()
      .reset_index(name="count")
)

bad_ids_future = future_counts.query("count != 12")

print("Bad IDs in df_future_full:", bad_ids_future["id"].tolist())

# Show counts for each bad ID
print("Counts for bad IDs (df_future_full):")
print(bad_ids_future)



Bad IDs in df_future_full: ['Kenya_Busia', 'Kenya_Gatare', 'Kenya_Mbale', 'Uganda_Kigoma', 'Uganda_Rubirizi']
Counts for bad IDs (df_future_full):
                    id  count
3468       Kenya_Busia    381
3695      Kenya_Gatare    381
4808       Kenya_Mbale    381
12782    Uganda_Kigoma    381
13463  Uganda_Rubirizi    381
Bad IDs in df_future_full: ['Kenya_Busia', 'Kenya_Gatare', 'Kenya_Mbale', 'Uganda_Kigoma', 'Uganda_Rubirizi']
Counts for bad IDs (df_future_full):
                    id  count
3468       Kenya_Busia     36
3695      Kenya_Gatare     36
4808       Kenya_Mbale     36
12782    Uganda_Kigoma     36
13463  Uganda_Rubirizi     36


In [7]:
#5x instances are duplicated 3x. Possibly from re-running cells in Preprocessing.ipynb. Just delete for now: 
df_context_full = df_context_full[~df_context_full["id"].isin(bad_ids_context["id"].tolist())]
df_future_full  = df_future_full[~df_future_full["id"].isin(bad_ids_future["id"].tolist())]


In [8]:
#Context:
context_counts = (
    df_context_full.groupby("id")
      .size()
      .reset_index(name="count")
)

bad_ids_context = context_counts.query("count != 127")

print("Bad IDs in df_future_full:", bad_ids_future["id"].tolist())

# Show counts for each bad ID
print("Counts for bad IDs (df_future_full):")
print(bad_ids_context)

#Future:
future_counts = (
    df_future_full.groupby("id")
      .size()
      .reset_index(name="count")
)

bad_ids_future = future_counts.query("count != 12")

print("Bad IDs in df_future_full:", bad_ids_future["id"].tolist())

# Show counts for each bad ID
print("Counts for bad IDs (df_future_full):")
print(bad_ids_future)



Bad IDs in df_future_full: ['Kenya_Busia', 'Kenya_Gatare', 'Kenya_Mbale', 'Uganda_Kigoma', 'Uganda_Rubirizi']
Counts for bad IDs (df_future_full):
Empty DataFrame
Columns: [id, count]
Index: []
Bad IDs in df_future_full: []
Counts for bad IDs (df_future_full):
Empty DataFrame
Columns: [id, count]
Index: []


## Preselecting contextual data for class balance

### Overall Class Breakdown: Raw observations

In [9]:
breakdown = (
    df_context_full['conflict_indicator']
      .value_counts()
      .sort_index()
      .rename_axis('conflict_indicator')
      .reset_index(name='count')
)

# Add percentage column
total = len(df_context_full)
breakdown['percentage'] = (breakdown['count'] / total * 100).round(2)

print(breakdown)


   conflict_indicator    count  percentage
0                   0  1689595       98.15
1                   1    29722        1.73
2                   2       20        0.00
3                   3     2148        0.12


### Id Breakdown by Maximum conflict_indicator

In [10]:
id_max = (
    df_context_full.groupby('id')['conflict_indicator']
      .max()
      .reset_index()
)

breakdown = (
    id_max['conflict_indicator']
        .value_counts()
        .sort_index()
        .rename_axis('conflict_indicator')
        .reset_index(name='count')
)

total_ids = id_max.shape[0]
breakdown['percentage'] = (breakdown['count'] / total_ids * 100).round(2)

print(breakdown)

   conflict_indicator  count  percentage
0                   0   5940       43.82
1                   1   7364       54.33
2                   2      1        0.01
3                   3    250        1.84


In [11]:
class_0 = (
    df_context_full
        .groupby("id")["conflict_indicator"]
        .max()
        .pipe(lambda s: s[s == 0].index)
)#.sample(n=200, random_state=42)

class_1 = (
    df_context_full
        .groupby("id")["conflict_indicator"]
        .max()
        .pipe(lambda s: s[s == 1].index)
)#.sample(n=200, random_state=42)
class_2 = (
    df_context_full
        .groupby("id")["conflict_indicator"]
        .max()
        .pipe(lambda s: s[s == 2].index)
)#.sample(n=200, random_state=42)
class_3 = (
    df_context_full
        .groupby("id")["conflict_indicator"]
        .max()
        .pipe(lambda s: s[s == 3].index)
)#.sample(n=200, random_state=42)

class_0 = pd.Series(class_0).sample(n = min([200, len(class_0)]), random_state = 42)#aggressivly undersample. 
class_1 = pd.Series(class_1).sample(n = min([200, len(class_1)]), random_state = 42)
class_2 = pd.Series(class_2).sample(n = min([200, len(class_2)]), random_state = 42)
class_3 = pd.Series(class_3).sample(n = min([200, len(class_3)]), random_state = 42)

context_df_stratified_sample_ids = list(set(class_0) | set(class_1) | set(class_2) | set(class_3))

#context_df_stratified_sample = pd.concat([class_0, class_1, class_2, class_3], ignore_index=True)
#context_ids = context_df_stratified_sample["id"].unique()

####future_df_sample = df_future_full[df_future_full["id"] = []]

print("Ids in stratified sample: ", len(context_df_stratified_sample_ids))

#select from the full datasets, then sort by series and time. 
context_df_stratified_sample = df_context_full[df_context_full['id'].isin(context_df_stratified_sample_ids)].sort_values(["id", "periodStart"])
future_df_stratified_sample = df_future_full[df_future_full['id'].isin(context_df_stratified_sample_ids)].sort_values(["id", "periodStart"])


Ids in stratified sample:  601


In [12]:
breakdown = (
    context_df_stratified_sample['conflict_indicator']
      .value_counts()
      .sort_index()
      .rename_axis('conflict_indicator')
      .reset_index(name='count')
)

# Add percentage column
total = len(context_df_stratified_sample)
breakdown['percentage'] = (breakdown['count'] / total * 100).round(2)

print(breakdown)


   conflict_indicator  count  percentage
0                   0  73271       96.00
1                   1   1305        1.71
2                   2     15        0.02
3                   3   1736        2.27


## Vanilla Model

In [13]:
#Install the amazon chronos package for chronos 2
!pip install chronos-forecasting

from chronos import Chronos2Pipeline

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Optional: restrict to one GPU
os.environ["CUDA_VISIBLE_DEVICES"] = "0"


### Load Vanilla model

In [14]:
#Load lora fnetuned model: downloads a fresh version of the vanilla weights and applies the saved weights. 
from chronos import Chronos2Pipeline

vanilla_chronos2_pipeline = Chronos2Pipeline.from_pretrained(
    "amazon/chronos-2",
    device_map="cpu"   #"CUDA also an option if you're a baller with a big GPU (I am not). 
)

print(type(vanilla_chronos2_pipeline))

Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

<class 'chronos.chronos2.pipeline.Chronos2Pipeline'>


In [15]:
# Generate predictions with covariates: takes roughly 4 mins. 

import datetime;

ct = datetime.datetime.now()
print("current time:", ct)

pred_df = vanilla_chronos2_pipeline.predict_df(
    context_df_stratified_sample,
    future_df=future_df_stratified_sample.drop('conflict_indicator', axis=1), #the sample future dataframe, with no target column . 
    prediction_length=12,  # Number of steps to forecast
    quantile_levels=[0.1, 0.5, 0.9],  # Quantiles for probabilistic forecast; 3x answers per test, model is 10%, 50% and 90% confident the value is below this answer. 
    id_column="id",  # Column identifying different time series
    timestamp_column="periodStart",  # Column with datetime information
    target="conflict_indicator",  # Column(s) with time series values to predict
    cross_learning=True,  # Enable cross-learning;
)
pred_df

ct = datetime.datetime.now()
print("current time:", ct)

current time: 2026-09-20 21:53:12.176113
current time: 2026-09-20 21:57:36.396180


### Making predictions & Evaluating

In [16]:
eval_df = pred_df.merge(
    future_df_stratified_sample[["id", "periodStart", "conflict_indicator"]],
    on=["id", "periodStart"],
    how="inner"
)
eval_df

,id,periodStart,target_name,predictions,0.1,0.5,0.9,conflict_indicator
0,Central_African_Republic_Abba,2024-09-26,conflict_indicator,0.001669,-0.007364,0.001669,0.907698,0
1,Central_African_Republic_Abba,2024-10-24,conflict_indicator,0.001560,-0.008094,0.001560,1.011537,0
2,Central_African_Republic_Abba,2024-11-21,conflict_indicator,0.001664,-0.007543,0.001664,1.046111,0
3,Central_African_Republic_Abba,2024-12-19,conflict_indicator,0.002821,-0.007688,0.002821,1.082156,0
4,Central_African_Republic_Abba,2025-01-16,conflict_indicator,0.002227,-0.007234,0.002227,1.191899,0
...,...,...,...,...,...,...,...,...
7207,Uganda_Yumbe,2025-04-10,conflict_indicator,0.003108,0.000349,0.003108,0.028242,0
7208,Uganda_Yumbe,2025-05-08,conflict_indicator,0.002080,0.000175,0.002080,0.028722,0
7209,Uganda_Yumbe,2025-06-05,conflict_indicator,0.003222,0.000608,0.003222,0.028952,0
7210,Uganda_Yumbe,2025-07-03,conflict_indicator,0.003082,0.000409,0.003082,0.025215,0


In [17]:
# NOW extract arrays
y_true = eval_df["conflict_indicator"].values
y_pred = eval_df["predictions"].values

#Regression metrics:
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# MSRE (Mean Squared Relative Error)
msre = ((y_true - y_pred)**2 / (y_true**2 + 1e-9)).mean()

# MSE
mse = mean_squared_error(y_true, y_pred)

# RMSE
rmse = rmse = mean_squared_error(y_true, y_pred) ** 0.5

# MAE
mae = mean_absolute_error(y_true, y_pred)

# R²
r2 = r2_score(y_true, y_pred)

print("MSRE:", msre)
print("MSE:", mse)
print("RMSE:", rmse)
print("MAE:", mae)
print("R²:", r2)

MSRE: 31087229.53392475
MSE: 0.19540190696716309
RMSE: 0.4420428791046894
MAE: 0.09132524579763412
R²: 0.0859026312828064


In [18]:
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error

#RMSE
y_true = eval_df["conflict_indicator"].values
y_pred = eval_df["predictions"].values

rmse = mean_squared_error(y_true, y_pred) ** 0.5


#MAE
mae = mean_absolute_error(y_true, y_pred)

#Pinball Loss
def pinball_loss(y_true, y_pred, q):
    return np.mean(np.maximum(q*(y_true - y_pred), (q-1)*(y_true - y_pred)))

pinball_01 = pinball_loss(y_true, eval_df["0.1"].values, 0.1)
pinball_05 = pinball_loss(y_true, eval_df["0.5"].values, 0.5)
pinball_09 = pinball_loss(y_true, eval_df["0.9"].values, 0.9)

#CRPS (Continuous Ranked Probability Score)
#CRPS can be approximated using quantiles:
#(Pinball losses integrated over quantiles)

quantiles = [0.1, 0.5, 0.9]
preds = [eval_df["0.1"].values, eval_df["0.5"].values, eval_df["0.9"].values]

# Approximate CRPS by averaging pinball losses across quantiles
crps = np.mean([
    pinball_loss(y_true, preds[i], quantiles[i])
    for i in range(len(quantiles))
])

# Print results
#print("(meaningless) RMSE:", rmse)
#print("(meaningless) MAE:", mae)
print("Pinball Loss (0.1):", pinball_01)
print("Pinball Loss (0.5):", pinball_05)
print("Pinball Loss (0.9):", pinball_09)
print("CRPS:", crps)


Pinball Loss (0.1): 0.009503992884251133
Pinball Loss (0.5): 0.045662625491725325
Pinball Loss (0.9): 0.06068925518119416
CRPS: 0.03861862451905687


In [19]:
#Add the cprs column calculated from the pinball values
import numpy as np

def pinball_loss(y_true, y_pred, q):
    diff = y_true - y_pred
    return np.maximum(q * diff, (q - 1) * diff)

# Compute CRPS from your quantile predictions
eval_df["crps"] = (
    pinball_loss(eval_df["conflict_indicator"], eval_df["0.1"], 0.1) +
    pinball_loss(eval_df["conflict_indicator"], eval_df["0.5"], 0.5) +
    pinball_loss(eval_df["conflict_indicator"], eval_df["0.9"], 0.9)
) / 3.0
eval_df

,id,periodStart,target_name,predictions,0.1,0.5,0.9,conflict_indicator,crps
0,Central_African_Republic_Abba,2024-09-26,conflict_indicator,0.001669,-0.007364,0.001669,0.907698,0,0.030780
1,Central_African_Republic_Abba,2024-10-24,conflict_indicator,0.001560,-0.008094,0.001560,1.011537,0,0.034248
2,Central_African_Republic_Abba,2024-11-21,conflict_indicator,0.001664,-0.007543,0.001664,1.046111,0,0.035399
3,Central_African_Republic_Abba,2024-12-19,conflict_indicator,0.002821,-0.007688,0.002821,1.082156,0,0.036798
4,Central_African_Republic_Abba,2025-01-16,conflict_indicator,0.002227,-0.007234,0.002227,1.191899,0,0.040342
...,...,...,...,...,...,...,...,...,...
7207,Uganda_Yumbe,2025-04-10,conflict_indicator,0.003108,0.000349,0.003108,0.028242,0,0.001564
7208,Uganda_Yumbe,2025-05-08,conflict_indicator,0.002080,0.000175,0.002080,0.028722,0,0.001357
7209,Uganda_Yumbe,2025-06-05,conflict_indicator,0.003222,0.000608,0.003222,0.028952,0,0.001685
7210,Uganda_Yumbe,2025-07-03,conflict_indicator,0.003082,0.000409,0.003082,0.025215,0,0.001477


### Per Class Evaluation

In [20]:
import pandas as pd

# Per-class CRPS
crps_per_class = eval_df.groupby("conflict_indicator")["crps"].mean()
print("CRPS per class:")
print(crps_per_class)

# Per-class pinball loss
pinball_01_per_class = eval_df.groupby("conflict_indicator")["0.1"].mean()
pinball_05_per_class = eval_df.groupby("conflict_indicator")["0.5"].mean()
pinball_09_per_class = eval_df.groupby("conflict_indicator")["0.9"].mean()

print("\nPinball Loss (0.1) per class:")
print(pinball_01_per_class)

print("\nPinball Loss (0.5) per class:")
print(pinball_05_per_class)

print("\nPinball Loss (0.9) per class:")
print(pinball_09_per_class)


CRPS per class:
conflict_indicator
0    0.014735
1    0.314667
2    0.438215
3    0.786029
Name: crps, dtype: float64

Pinball Loss (0.1) per class:
conflict_indicator
0   -0.001347
1    0.027833
2   -0.022741
3    0.190134
Name: 0.1, dtype: float32

Pinball Loss (0.5) per class:
conflict_indicator
0    0.022205
1    0.266678
2    0.010415
3    0.706257
Name: 0.5, dtype: float32

Pinball Loss (0.9) per class:
conflict_indicator
0    0.324324
1    1.003504
2    3.175788
3    2.976971
Name: 0.9, dtype: float32


### Analysis

We can see CRPS is collapsing in the minority classes.

* Class 0: the model is extremely accurate. CRPS ≈ 0.015 is very strong.
* Class 1: Predictions are OK but noticeably worse. Likely Ok for this application. 
* Class 2: Model struggles; uncertainty is high, not unexpected as this is very much a minority class. Only 1x Id has a max of conflict_indicator = 2. 
* Class 3: Model performs worst; distribution is wide and inaccurate. This is a surprise, possibly again down to rarity over all observations within the class. 

---

* Class 0: Very tight, accurate lower‑tail predictions.
* Class 1: Slightly too low.
* Class 2: Slightly too high.
* Class 3: Lower quantile is badly underestimated → model thinks the lower tail is much lower than reality.

Pinball Loss 0.5:
* Class 0: Median predictions are very accurate.
* Class 1: Median is off by a moderate amount.
* Class 2: Surprisingly good median accuracy.
* Class 3: Median is badly off → model struggles to place the central tendency.

Pinball loss 0.9:
* Class 0: Upper tail is reasonably predicted.
* Class 1: Upper tail is off by ~1 unit.
* Class 2: Upper tail is extremely off.
* Class 3: Upper tail also extremely off. The model cannot predict extreme high‑conflict outcomes well.
This is very typical- the upper tail is the hardest part of the distribution to learn, especially for rare events.

### Final analysis based on that strategy: Not great, not terrible. 

Next steps:
* Class 2 (Regional conflict only, no local conflict) is very rare. We can collapse this into class 3 for three values of conflict_indicator.
* We can refine the sampling strategy. We can see that the class 0 instances are still 94% of the dataset by observation.

 

### Refining the Dataset


In [21]:
#Median is used instead of mean as it is robust to outliers.  
#Top Ids for local indicators (extremely dangerous):
mean_local_fatalities = (
    df_context_full.groupby("id")["LocalTotalFatalities"]
      .median()
      .sort_values(ascending=False)
)
top_ids_local_fatalities = mean_local_fatalities.head(10)


mean_local_avg_severity = (
    df_context_full.groupby("id")["LocalAvgSeverity"]
      .median()
      .sort_values(ascending=False)
)
top_ids_local_avg_severity = mean_local_avg_severity.head(10)

#Top Ids for regional indicators (extremely dangerous):
mean_regional_fatalities = (
    df_context_full.groupby("id")["RegionalTotalFatalities"]
      .median()
      .sort_values(ascending=False)
)
top_ids_regional_fatalities = mean_regional_fatalities.head(10)


mean_regional_avg_severity = (
    df_context_full.groupby("id")["RegionalAvgSeverity"]
      .median()
      .sort_values(ascending=False)
)
top_ids_regional_avg_severity = mean_regional_avg_severity.head(10)
print(top_ids_regional_avg_severity) #many of these are places in Mogadishu- the same events will be shown as within 10k. 
#print(pd.Series(top_ids_regional_avg_severity) + pd.Series(top_ids_local_fatalities))

id
Somalia_Afgooye          3.375000
Somalia_Marka            3.000000
Somalia_Belet_Weyne      2.789474
Somalia_Bulo_Burto       2.600000
Somalia_Jalalaqsi        2.500000
Somalia_Jowhar           2.500000
Somalia_Gaalkacyo        2.000000
Somalia_Dhuusamarreeb    2.000000
Somalia_Jilib            2.000000
Sudan_Tawila             1.666667
Name: RegionalAvgSeverity, dtype: float64


In [22]:
nonzero_local_counts = (
    df_context_full[df_context_full["LocalTotalFatalities"] > 0]
      .groupby("id")["LocalTotalFatalities"]
      .count()
      .sort_values(ascending=False)
)
top_ids_nonzero_local = nonzero_local_counts.head(10)


df_context_full["rolling_std"] = (
    df_context_full.groupby("id")["LocalTotalFatalities"]
      .rolling(window=6, min_periods=1)
      .std()
      .reset_index(level=0, drop=True)
)

rolling_std_rank = (
    df_context_full.groupby("id")["rolling_std"]
      .mean()
      .sort_values(ascending=False)
)

top_ids_rolling_std = rolling_std_rank.head(10)


## LoRA Training

### Set up the LoRA Run

Low-rank adapters. Trains the attention heads and feedforwards layers by tweaking matrices- not full model training. In theory this will make the model more adapted to our specific task. Takes ~30 mins for 1000 steps, assuming it doesn't fail after 853, which has happened. 

In [24]:
from chronos.chronos2 import preprocess

ct = datetime.datetime.now()
print("current time:", ct)


# Prepare data for fine-tuning using the retail sales dataset
train_inputs_lora = preprocess.from_data_frame(
    context_df_stratified_sample,
    target_columns=["conflict_indicator"],
    prediction_length=12,
    id_column="id",
    timestamp_column="periodStart",
    known_covariates_names=[ "name",	"country",	"latitude",	"longitude",	"minBorderDistanceKm",	"minCapitalDistanceKm", "is_voting"], #Knowable values in the future dataset. Remaining columns are treated as past-only covariates
)
print("executed train_inputs step")

# Load Chronos‑2
pipeline_lora = Chronos2Pipeline.from_pretrained(
    "amazon/chronos-2",
    device_map="cpu"   #"CUDA also an option if you're a baller with a big GPU (I am not). 
)

# Fine-tune the model with LoRA
lora_chronos_pipeline = pipeline_lora.fit(
    inputs=train_inputs_lora,
    prediction_length=12, #must match the number of observations in the future_df per place. 
    num_steps=800,#Set to 10 just to check that it works. 
    learning_rate=1e-4,
    batch_size=32,
    logging_steps=100,
    finetune_mode="lora"
)
print("executed pipeline_lora.fit step")

ct = datetime.datetime.now()
print("current time:", ct)

current time: 2026-09-20 21:59:55.645912
executed train_inputs step


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Step,Training Loss
100,0.029969
200,0.018411
300,0.049573
400,0.041871
500,0.041422
600,0.027464
700,0.037178
800,0.026781


executed pipeline_lora.fit step
current time: 2026-09-20 22:22:14.974261


In [26]:
#Save the finetuned model
print(type(lora_chronos_pipeline))
lora_chronos_pipeline.save_pretrained("lora_chronos_model")

<class 'chronos.chronos2.pipeline.Chronos2Pipeline'>


In [30]:
#Takes ~15 mins on the same dataset. 
 ct = datetime.datetime.now()
print("current time:", ct)

pred_df = lora_chronos_pipeline.predict_df(
    context_df_stratified_sample,
    future_df=future_df_stratified_sample.drop('conflict_indicator', axis=1), #the sample future dataframe, with no target column . 
    prediction_length=12,  # Number of steps to forecast
    quantile_levels=[0.1, 0.5, 0.9],  # Quantiles for probabilistic forecast; 3x answers per test, model is 10%, 50% and 90% confident the value is below this answer. 
    id_column="id",  # Column identifying different time series
    timestamp_column="periodStart",  # Column with datetime information
    target="conflict_indicator",  # Column(s) with time series values to predict
    cross_learning=True,  # Enable cross-learning;
)
pred_df

ct = datetime.datetime.now()
print("current time:", ct)

current time: 2026-09-20 22:25:41.832183
current time: 2026-09-20 22:40:46.815222


### Making predictions & Evaluating

In [31]:
eval_df = pred_df.merge(
    future_df_stratified_sample[["id", "periodStart", "conflict_indicator"]],
    on=["id", "periodStart"],
    how="inner"
)
eval_df

,id,periodStart,target_name,predictions,0.1,0.5,0.9,conflict_indicator
0,Central_African_Republic_Abba,2024-09-26,conflict_indicator,-0.005650,-0.033930,-0.005650,0.082946,0
1,Central_African_Republic_Abba,2024-10-24,conflict_indicator,-0.004962,-0.011357,-0.004962,0.092997,0
2,Central_African_Republic_Abba,2024-11-21,conflict_indicator,-0.003417,-0.011558,-0.003417,0.090062,0
3,Central_African_Republic_Abba,2024-12-19,conflict_indicator,-0.002207,-0.010153,-0.002207,0.111390,0
4,Central_African_Republic_Abba,2025-01-16,conflict_indicator,0.010180,-0.012179,0.010180,0.125725,0
...,...,...,...,...,...,...,...,...
7207,Uganda_Yumbe,2025-04-10,conflict_indicator,0.001687,-0.000324,0.001687,0.004794,0
7208,Uganda_Yumbe,2025-05-08,conflict_indicator,0.002402,0.000416,0.002402,0.005045,0
7209,Uganda_Yumbe,2025-06-05,conflict_indicator,0.002176,0.000416,0.002176,0.005447,0
7210,Uganda_Yumbe,2025-07-03,conflict_indicator,0.002452,0.000377,0.002452,0.004481,0


### Per Class Evaluation

In [32]:
#Add the cprs column calculated from the pinball values
import numpy as np

def pinball_loss(y_true, y_pred, q):
    diff = y_true - y_pred
    return np.maximum(q * diff, (q - 1) * diff)

# Compute CRPS from your quantile predictions
eval_df["crps"] = (
    pinball_loss(eval_df["conflict_indicator"], eval_df["0.1"], 0.1) +
    pinball_loss(eval_df["conflict_indicator"], eval_df["0.5"], 0.5) +
    pinball_loss(eval_df["conflict_indicator"], eval_df["0.9"], 0.9)
) / 3.0
eval_df

,id,periodStart,target_name,predictions,0.1,0.5,0.9,conflict_indicator,crps
0,Central_African_Republic_Abba,2024-09-26,conflict_indicator,-0.005650,-0.033930,-0.005650,0.082946,0,0.004838
1,Central_African_Republic_Abba,2024-10-24,conflict_indicator,-0.004962,-0.011357,-0.004962,0.092997,0,0.004306
2,Central_African_Republic_Abba,2024-11-21,conflict_indicator,-0.003417,-0.011558,-0.003417,0.090062,0,0.003957
3,Central_African_Republic_Abba,2024-12-19,conflict_indicator,-0.002207,-0.010153,-0.002207,0.111390,0,0.004419
4,Central_African_Republic_Abba,2025-01-16,conflict_indicator,0.010180,-0.012179,0.010180,0.125725,0,0.006294
...,...,...,...,...,...,...,...,...,...
7207,Uganda_Yumbe,2025-04-10,conflict_indicator,0.001687,-0.000324,0.001687,0.004794,0,0.000452
7208,Uganda_Yumbe,2025-05-08,conflict_indicator,0.002402,0.000416,0.002402,0.005045,0,0.000693
7209,Uganda_Yumbe,2025-06-05,conflict_indicator,0.002176,0.000416,0.002176,0.005447,0,0.000669
7210,Uganda_Yumbe,2025-07-03,conflict_indicator,0.002452,0.000377,0.002452,0.004481,0,0.000671


In [33]:

# Per-class CRPS
crps_per_class = eval_df.groupby("conflict_indicator")["crps"].mean()
print("CRPS per class:")
print(crps_per_class)

# Per-class pinball loss
pinball_01_per_class = eval_df.groupby("conflict_indicator")["0.1"].mean()
pinball_05_per_class = eval_df.groupby("conflict_indicator")["0.5"].mean()
pinball_09_per_class = eval_df.groupby("conflict_indicator")["0.9"].mean()

print("\nPinball Loss (0.1) per class:")
print(pinball_01_per_class)

print("\nPinball Loss (0.5) per class:")
print(pinball_05_per_class)

print("\nPinball Loss (0.9) per class:")
print(pinball_09_per_class)


CRPS per class:
conflict_indicator
0    0.009091
1    0.327964
2    0.435247
3    0.830050
Name: crps, dtype: float64

Pinball Loss (0.1) per class:
conflict_indicator
0   -0.002423
1    0.007189
2   -0.044386
3    0.135221
Name: 0.1, dtype: float32

Pinball Loss (0.5) per class:
conflict_indicator
0    0.016791
1    0.198582
2   -0.004101
3    0.615936
Name: 0.5, dtype: float32

Pinball Loss (0.9) per class:
conflict_indicator
0    0.171714
1    0.715707
2    2.992509
3    2.194379
Name: 0.9, dtype: float32


### Analysis post LoRA: It's gotten worse at predicting the minority classes!